# History Spotify User Silver 🥈
---

## Change logs

| Version | Description                | Update Date |
|---------|----------------------------|-------------|
| 1.0     | Silver Table completed     | 2026-06-23  |


## Import Libs

In [0]:
import pandas as pd
import glob
from dotenv import load_dotenv
import os
from pyspark.sql.functions import col,when,lower,initcap,expr,to_timestamp, from_utc_timestamp,hour,date_format

## Load Env

In [0]:
load_dotenv()

## Read Bronze Table

In [0]:
df_audio_01 = spark.table(f"{os.getenv("CATALOG")}.{os.getenv("BRONZE_SCHEMA")}.{os.getenv("BRONZE_NAME_TABLE_AUDIO")}")

## Remove colluns not used

In [0]:
drop_columns = [
    'ip_addr',
    'spotify_track_uri',
    'spotify_episode_uri',
    'conn_country',
    'audiobook_uri',
    'audiobook_chapter_uri',
    'audiobook_chapter_uri',
    'offline',
    'offline_timestamp',
    'incognito_mode'
]

df_audio_02 = df_audio_01.drop(*drop_columns)

## Convert column TS UTC+0 to UTC-3

In [0]:
df_audio_03 = df_audio_02.withColumn(
    "ts",
    from_utc_timestamp(to_timestamp(col("ts")), "America/Sao_Paulo")
)

## Change type and rename columns

In [0]:
df_audio_04 = df_audio_03.select(
    col("ts").cast("timestamp").alias("ts_play"),
    col("platform").alias("nm_platform"),
    col("ms_played").alias("nr_ms_played"),
    initcap(col("master_metadata_track_name")).alias("nm_track"),
    col("master_metadata_album_artist_name").alias("nm_artist"),
    col("master_metadata_album_album_name").alias("nm_album"),
    col("episode_name").alias("nm_episode"),
    col("episode_show_name").alias("nm_show"),
    col("audiobook_title").alias("nm_audiobook"),
    col("audiobook_chapter_title").alias("nm_audiobook_chapter"),
    col("reason_start").alias("nm_reason_start"),
    col("reason_end").alias("nm_reason_end"),
    col("shuffle").alias("fl_shuffle"),
    col("skipped").alias("fl_skipped")
)

## Rename result nm_platform

In [0]:
df_audio_05 = df_audio_04.withColumn(
    "nm_platform",
    when(
        lower(col("nm_platform")).contains("windows"), "Windows"
    ).when(
        lower(col("nm_platform")).contains("android"), "Android"
    ).when(
        lower(col("nm_platform")).contains("ios"), "IOS"
    ).when(
        lower(col("nm_platform")).contains("mac"), "Mac"
    ).when(
        lower(col("nm_platform")).contains("linux"), "Linux"
    ).when(
        lower(col("nm_platform")).contains("web"), "Web"
    ).when(
        lower(col("nm_platform")).contains("amazon"), "Alexa"
    ).otherwise("Not define")
)

## Create a column of hour play

In [0]:
df_audio_06 = df_audio_05.withColumn(
    "tm_hour_play",
    expr("date_format(ts_play, 'HH:mm:ss')")
)

## Create a column of period of day play

In [0]:
df_audio_07 = df_audio_06.withColumn(
    "nm_period_play",
    when(
        (hour(col("ts_play")) >= 0) & (hour(col("ts_play")) < 6), "Dawn"
    ).when(
        (hour(col("ts_play")) >= 6) & (hour(col("ts_play")) < 12), "Morning"
    ).when(
        (hour(col("ts_play")) >= 12) & (hour(col("ts_play")) < 18), "Afternoon"
    ).when(
        (hour(col("ts_play")) >= 18) & (hour(col("ts_play")) < 24), "Night"
    ).otherwise("Undefined")
)

## Save dataframe

In [0]:
df_audio_07.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{os.getenv("CATALOG")}.{os.getenv("SILVER_SCHEMA")}.{os.getenv("SILVER_NAME_TABLE_AUDIO")}")